# SEMAINE 2 - ETAPE 3 : RECUPERATION DES LIGNES
*Marie Bizais*

*On identifie les lignes sur les pages, on les ordonne de haut en bas, puis on lance une reconnaissance du texte en français.*

## Préparation de l'environnement
### Les librairies

In [1]:
from ultralytics import YOLO

In [2]:
mymodel = YOLO('./entrainement/weights/best.pt') # le meilleur modèle obtenu pendant l'entraînement

In [19]:
from PIL import Image
from pathlib import Path
import shutil

### Fonctions

In [2]:
def parse_yolo_labels(label_path, target_class): #Retourne la liste des bbox de la classe cible, triées par cy (centroïde) croissant.
    bboxes = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            if cls != target_class:
                continue
            cx, cy, w, h = map(float, parts[1:5])
            bboxes.append((cy, cx, w, h))   # cy en premier pour le tri
    bboxes.sort(key=lambda b: b[0])     # Tri par y_centre croissant = haut de page → bas de page

    return bboxes

In [10]:
def crop_sorted_boxes(image_path, bboxes, output_dir, label): #Croppe chaque bbox dans l'ordre de la liste et sauvegarde les fichiers sous la forme :  <stem>_<label><N><ext>   (ex. ma_page_Verse1.jpg)
    img = Image.open(image_path)
    W, H = img.size
    stem = image_path.stem
    ext  = image_path.suffix
    out  = Path(output_dir) if output_dir else image_path.parent
    out.mkdir(parents=True, exist_ok=True)
 
    saved = []
    if len(bboxes) == 0:
        print(f"[{image_path.name}] Aucune bbox de classe {label} trouvée.")
    else:
        for i, (cy, cx, w, h) in enumerate(bboxes, 1):
            x1 = max(0, int((cx - w / 2) * W))
            y1 = max(0, int((cy - h / 2) * H))
            x2 = min(W, int((cx + w / 2) * W))
            y2 = min(H, int((cy + h / 2) * H))
 
            crop = img.crop((x1, y1, x2, y2))
            filename = out / f"{stem}_{label}{i}{ext}"
            crop.save(filename)
            saved.append(filename)
            print(f"  {label}{i:>3} → ({x1}, {y1}, {x2}, {y2})  →  {filename.name}")
    
        print(f"\n✓ {len(saved)} crop(s) sauvegardé(s) dans : {out}/")
    return saved

In [ ]:
# def yolo_to_pixels(cx, cy, w, h, img_w, img_h): #Convertit les coordonnées YOLO normalisées en pixels (x1, y1, x2, y2)
#     x1 = int((cx - w / 2) * img_w)
#     y1 = int((cy - h / 2) * img_h)
#     x2 = int((cx + w / 2) * img_w)
#     y2 = int((cy + h / 2) * img_h)
#     # Clamp pour rester dans l'image
#     x1, y1 = max(0, x1), max(0, y1)
#     x2, y2 = min(img_w, x2), min(img_h, y2)
#     return x1, y1, x2, y2

In [ ]:
# def process(image_path, label_path, target_class):
#     img = Image.open(image_path)
#     img_w, img_h = img.size
 
#     boxes = parse_yolo_labels(label_path, target_class)
#     if not boxes:
#         print(f"[{image_path.name}] Aucune bbox de classe {target_class} trouvée.")
#         return
 
#     stem = image_path.stem
#     out_dir = image_path.parent
#     suffix = image_path.suffix  # conserve le format d'origine (.jpg, .png…)
 
#     print(f"\n[{image_path.name}]  {len(boxes)} bbox(s) classe {target_class} trouvée(s)")
 
#     for idx, (cy, cx, w, h) in enumerate(boxes, start=1):
#         x1, y1, x2, y2 = yolo_to_pixels(cx, cy, w, h, img_w, img_h)
#         crop = img.crop((x1, y1, x2, y2))
#         out_name = f"{stem}_Verse{idx}{suffix}"
#         out_path = out_dir / out_name
#         crop.save(out_path)
#         print(f"  Verse{idx:>3} → y_centre={cy:.4f}  pixels=({x1},{y1},{x2},{y2})  → {out_name}")
 
#     print(f"  {len(boxes)} crop(s) exporté(s) dans {out_dir}/")


### Fichiers et arguments

In [17]:
SOURCE_DIR = "../DATA/pages/Corm_montagne"
exemple = exemple = '../DATA/pages/Corm_montagne/image_34.jpg'
PRED_DIR     = Path("./predictions/my_model")
LABELS_DIR   = PRED_DIR / "labels"
IMAGES_DIR   = PRED_DIR / "images"   # ← sera créé
OUTPUT_DIR   = PRED_DIR / "crops"
TARGET_CLASS = 9
LABEL        = "Verse"
SUFFIXES = [".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG"]

## Identification des lignes
On lance l'identification avec le modèle que nous avons produit sur l'ensemble des images du livre.

In [ ]:
results = mymodel.predict(source=exemple, imgsz=1024, device='0', save=True, save_txt=True, project='./predictions', name='my_model', exist_ok=True) # on peut aussi envisager une boucle sur SOURCE_DIR, quand on est prêt.


image 1/1 /home/bizais/Documents/Distam/2026/cormatusj2026/WEEK_2/../DATA/pages/Corm_montagne/image_34.jpg: 1024x704 7 LigneDeProses, 2 N°Pages, 39 Verss, 59.6ms
Speed: 5.0ms preprocess, 59.6ms inference, 10.3ms postprocess per image at shape (1, 3, 1024, 704)
Results saved to /home/bizais/Documents/Distam/2026/cormatusj2026/WEEK_2/predictions/my_model
1 label saved to /home/bizais/Documents/Distam/2026/cormatusj2026/WEEK_2/predictions/my_model/labels


*On fait un peu de réorganisation des fichiers*

In [ ]:
#IMAGES_DIR.mkdir(exist_ok=True)
# for suffix in SUFFIXES:
#     for image_path in PRED_DIR.glob(f"*{suffix}"):
#         shutil.move(str(image_path), IMAGES_DIR / image_path.name)
# print(f"Images déplacées vers {IMAGES_DIR}/")

Images déplacées vers predictions/my_model/images/


In [ ]:
IMAGES_DIR.mkdir(exist_ok=True) # on évite la double boucle
for image_path in PRED_DIR.iterdir():
    if image_path.suffix.lower() in SUFFIXES:
        shutil.move(str(image_path), IMAGES_DIR / image_path.name)

## Ordonnancement
On récupère les coordonnées des différents rectangles qui délimitent les lignes et on les classe de haut en bas.

*Pour simplifier les choses, nous abandonnons les titres courants et numéros de page, et nous n'utilisons pas les types de lignes. Une option plus riche consisterait à travailler sur des fichiers qui conserveraient l'information des types, ce qui nous servirait ensuite pour structurer le texte. C'est un sujet que nous aborderons lors de la troisième séance.*

In [ ]:
parse_yolo_labels('./predictions/my_model/labels/image_34.txt', 9) # mettre à jour avec les arguments préparés en amont : il n'y aura plus besoin de modifier ici.

[(0.0980131, 0.781634, 0.258538, 0.0237758),
 (0.132693, 0.579157, 0.188529, 0.019768),
 (0.153298, 0.581773, 0.15185, 0.0165045),
 (0.172032, 0.580756, 0.185482, 0.018822),
 (0.193147, 0.584312, 0.315974, 0.0197525),
 (0.211371, 0.607255, 0.165673, 0.0200639),
 (0.212067, 0.58021, 0.202732, 0.0211816),
 (0.212316, 0.572617, 0.287966, 0.0210287),
 (0.232055, 0.581593, 0.304517, 0.0188081),
 (0.255103, 0.579752, 0.315097, 0.0237431),
 (0.350747, 0.606582, 0.499687, 0.0216104),
 (0.373551, 0.597197, 0.444456, 0.0206453),
 (0.396061, 0.590537, 0.425367, 0.02158),
 (0.42568, 0.611916, 0.454946, 0.0225099),
 (0.448565, 0.609249, 0.447588, 0.0228966),
 (0.478113, 0.577688, 0.468999, 0.0196061),
 (0.500802, 0.601397, 0.462407, 0.0217274),
 (0.521144, 0.593072, 0.522102, 0.0184932),
 (0.522727, 0.631692, 0.396623, 0.0209388),
 (0.54222, 0.585176, 0.206637, 0.0226049),
 (0.569918, 0.586028, 0.494135, 0.0195546),
 (0.573818, 0.581688, 0.470802, 0.019591),
 (0.595974, 0.580263, 0.255454, 0.020943

## Récupération des images des zones listées
Nous disposons à présent des coordonnées des lignes qui correspondent à des vers sur notre page, triées de haut en bas. Nous les enregistrons dans l'ordre en reprenant le nom du fichier d'origine suivi de "Verses" puis du chiffre correspondant à l'ordre.

In [ ]:
# saved_files = crop_sorted_boxes(exemple, bboxes, "./cropped", "Verses")

NameError: name 'bboxes' is not defined

In [ ]:
#for image_path in sorted(IMAGES_DIR.glob(f"*{suffix}" for suffix in SUFFIXES)):
    label_path = LABELS_DIR / image_path.with_suffix(".txt").name

    if not label_path.exists():
        print(f"[SKIP] Label introuvable : {label_path.name}")
        continue

    sorted_boxes = load_sorted_boxes(label_path, TARGET_CLASS)
    crop_sorted_boxes(image_path, sorted_boxes, OUTPUT_DIR, LABEL)

TypeError: argument should be a str or an os.PathLike object where __fspath__ returns a str, not 'generator'

In [24]:
for file in IMAGES_DIR.iterdir():
    if file.suffix.lower() not in SUFFIXES:
        continue

    label_path = LABELS_DIR / file.with_suffix(".txt").name

    if not label_path.exists():
        print(f"[SKIP] Label introuvable : {label_path.name}")
        continue

    sorted_boxes = parse_yolo_labels(label_path, TARGET_CLASS)
    crop_sorted_boxes(file, sorted_boxes, OUTPUT_DIR, LABEL)
    print(f"{file.name} processed")

  Verse  1 → (1847, 366, 2579, 467)  →  image_34_Verse1.jpg
  Verse  2 → (1373, 522, 1907, 606)  →  image_34_Verse2.jpg
  Verse  3 → (1432, 617, 1862, 687)  →  image_34_Verse3.jpg
  Verse  4 → (1382, 692, 1907, 772)  →  image_34_Verse4.jpg
  Verse  5 → (1207, 780, 2102, 864)  →  image_34_Verse5.jpg
  Verse  6 → (1485, 856, 1954, 942)  →  image_34_Verse6.jpg
  Verse  7 → (1356, 857, 1930, 947)  →  image_34_Verse7.jpg
  Verse  8 → (1213, 858, 2029, 948)  →  image_34_Verse8.jpg
  Verse  9 → (1215, 947, 2078, 1027)  →  image_34_Verse9.jpg
  Verse 10 → (1195, 1035, 2088, 1136)  →  image_34_Verse10.jpg
  Verse 11 → (1010, 1446, 2425, 1538)  →  image_34_Verse11.jpg
  Verse 12 → (1061, 1545, 2320, 1633)  →  image_34_Verse12.jpg
  Verse 13 → (1070, 1639, 2274, 1731)  →  image_34_Verse13.jpg
  Verse 14 → (1088, 1763, 2377, 1859)  →  image_34_Verse14.jpg
  Verse 15 → (1091, 1860, 2359, 1957)  →  image_34_Verse15.jpg
  Verse 16 → (971, 1993, 2300, 2076)  →  image_34_Verse16.jpg
  Verse 17 → (1048,